# Task 6: House Price Prediction
## DevelopersHub Corporation — AI/ML Engineering Internship

**Name:** Muhammad Hassan Murad  
**DHC ID:** DHC-2360

---

### Objective
Predict house prices using property features such as size, bedrooms, location, and condition.

### Dataset
**California Housing Dataset** — available via scikit-learn (no Kaggle account needed).  
20,640 California census blocks with housing data.

### Models
- **Linear Regression** — baseline
- **Gradient Boosting Regressor** — powerful ensemble method

### Evaluation Metrics
- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- R² Score

---

## 1. Setup & Dependencies

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn --quiet
print("✅ Libraries installed.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-darkgrid')
COLORS = {'actual': '#2196F3', 'lr': '#FF5722', 'gb': '#4CAF50', 'accent': '#9C27B0'}

print("✅ All imports successful.")

## 2. Load Dataset

In [ ]:
# Load California Housing — no download required
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['Price'] = housing.target  # Median house value in $100,000s

print(f"✅ Dataset loaded: {df.shape[0]:,} samples, {df.shape[1]} columns")
print(f"\nFeatures: {housing.feature_names}")
print(f"Target  : Median house value (in $100,000s)")
print(f"Price range: ${df['Price'].min():.2f} — ${df['Price'].max():.2f} ($100k units)")
print(f"            = ${df['Price'].min()*100:.0f}k — ${df['Price'].max()*100:.0f}k")

df.head()

### Feature Dictionary

| Feature | Description |
|---|---|
| `MedInc` | Median income in block (tens of thousands USD) |
| `HouseAge` | Median house age in block (years) |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | Block population |
| `AveOccup` | Average number of household members |
| `Latitude` | Block latitude |
| `Longitude` | Block longitude |
| `Price` | **Target: Median house value ($100,000s)** |

## 3. Data Inspection & Cleaning

In [ ]:
print("=" * 50)
print("DESCRIPTIVE STATISTICS")
print("=" * 50)
print(df.describe().round(2).to_string())

print("\nMissing Values:")
missing = df.isnull().sum()
print(missing if missing.any() else "  None ✅")

In [ ]:
# Cap extreme outliers in AveRooms and AveBedrms (data entry artifacts)
print("Outlier Treatment:")
for col, cap in [('AveRooms', 20), ('AveBedrms', 6), ('AveOccup', 10)]:
    n_outliers = (df[col] > cap).sum()
    df[col] = df[col].clip(upper=cap)
    print(f"  {col}: capped {n_outliers} values at {cap}")

print("\n✅ Outlier treatment complete.")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ── House Price Distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df['Price'], bins=50, color=COLORS['actual'], alpha=0.8, edgecolor='white')
axes[0].axvline(df['Price'].mean(), color='black', linestyle='--', linewidth=2,
                label=f'Mean: ${df["Price"].mean()*100:.0f}k')
axes[0].axvline(df['Price'].median(), color='red', linestyle='--', linewidth=2,
                label=f'Median: ${df["Price"].median()*100:.0f}k')
axes[0].set_title('House Price Distribution', fontweight='bold')
axes[0].set_xlabel('Price ($100k)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Log-transformed price
axes[1].hist(np.log1p(df['Price']), bins=50, color=COLORS['accent'], alpha=0.8, edgecolor='white')
axes[1].set_title('Log-Transformed Price Distribution', fontweight='bold')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Frequency')

plt.suptitle('California Housing — Price Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Insight: Price is right-skewed — many affordable homes, fewer very expensive ones.")
print("Prices are capped at $500k (5.0) in this dataset.")

In [ ]:
# ── Feature vs Price Scatter Plots ─────────────────────────────────────────────
key_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms']
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    sc = ax.scatter(df[feat], df['Price'],
                    alpha=0.15, s=8, c=df['Price'],
                    cmap='viridis', edgecolors='none')
    plt.colorbar(sc, ax=ax, label='Price ($100k)')
    # Add trend line
    z = np.polyfit(df[feat], df['Price'], 1)
    p = np.poly1d(z)
    x_sorted = np.sort(df[feat].values)
    ax.plot(x_sorted, p(x_sorted), 'r-', linewidth=2, alpha=0.8, label='Trend')
    corr = df[feat].corr(df['Price'])
    ax.set_title(f'{feat} vs Price (r = {corr:.2f})', fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Price ($100k)')

plt.suptitle('Feature vs House Price Relationships', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_feature_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Geographic Price Map ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

sc = ax.scatter(df['Longitude'], df['Latitude'],
                c=df['Price'], cmap='YlOrRd',
                s=df['Population'] / df['Population'].max() * 30,
                alpha=0.5, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Median House Price ($100k)')
ax.set_title('California House Prices by Geographic Location',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.text(-118.5, 33.7, 'Los Angeles', fontsize=9, color='navy', fontweight='bold')
ax.text(-122.5, 37.5, 'San Francisco', fontsize=9, color='navy', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_geographic_map.png', dpi=150, bbox_inches='tight')
plt.show()

print("Insight: Coastal areas (San Francisco, LA) have significantly higher prices.")

In [ ]:
# ── Correlation Heatmap ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))

corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
            mask=mask, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 9})
ax.set_title('Feature Correlation Matrix (incl. Price)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top correlations with Price:")
print(corr['Price'].drop('Price').sort_values(key=abs, ascending=False).round(3).to_string())

## 5. Feature Engineering

In [ ]:
# Engineer additional meaningful features
df['RoomsPerPerson']    = df['AveRooms']  / df['AveOccup']   # Space per person
df['BedroomRatio']      = df['AveBedrms'] / df['AveRooms']   # Proportion of bedrooms
df['PopDensity']        = df['Population'] / df['AveOccup']  # Households per block

FEATURE_COLS = [
    'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
    'Population', 'AveOccup', 'Latitude', 'Longitude',
    'RoomsPerPerson', 'BedroomRatio', 'PopDensity'
]

print(f"✅ Feature engineering complete.")
print(f"   Features: {FEATURE_COLS}")
print(f"   Total: {len(FEATURE_COLS)} features")

## 6. Train / Test Split

In [ ]:
X = df[FEATURE_COLS].values
y = df['Price'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"\n✅ Data ready for modelling.")

## 7. Model Training

In [ ]:
# ── Linear Regression (Baseline) ──────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
print("✅ Linear Regression trained.")

# ── Gradient Boosting ─────────────────────────────────────────────────────────
gb = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.08,
    max_depth=5,
    min_samples_leaf=5,
    subsample=0.8,
    random_state=42
)
gb.fit(X_train_sc, y_train)
y_pred_gb = gb.predict(X_test_sc)
print("✅ Gradient Boosting trained.")

## 8. Model Evaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    # Convert to dollars for interpretability
    mae_usd  = mae * 100_000
    rmse_usd = rmse * 100_000
    return {
        'Model'    : name,
        'MAE'      : mae,
        'MAE ($)'  : f'${mae_usd:,.0f}',
        'RMSE'     : rmse,
        'RMSE ($)' : f'${rmse_usd:,.0f}',
        'R²'       : r2,
    }

results = pd.DataFrame([
    evaluate('Linear Regression',   y_test, y_pred_lr),
    evaluate('Gradient Boosting',   y_test, y_pred_gb),
])

print("📊 Model Evaluation Metrics")
print("=" * 65)
print(results[['Model', 'MAE ($)', 'RMSE ($)', 'R²']].to_string(index=False))
print("\nNote: MAE and RMSE are in both $100k units and real USD for clarity.")

## 9. Visualizations

In [ ]:
# ── Actual vs Predicted ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, preds, name, color in zip(
    axes,
    [y_pred_lr, y_pred_gb],
    ['Linear Regression', 'Gradient Boosting'],
    [COLORS['lr'], COLORS['gb']]
):
    ax.scatter(y_test, preds, alpha=0.25, s=10, color=color, edgecolors='none')
    # Perfect prediction line
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, 'k--', linewidth=2, label='Perfect Prediction', zorder=5)

    mae = mean_absolute_error(y_test, preds)
    r2  = r2_score(y_test, preds)
    ax.set_title(f'{name}\nMAE: ${mae*100_000:,.0f} | R²: {r2:.3f}', fontweight='bold')
    ax.set_xlabel('Actual Price ($100k)')
    ax.set_ylabel('Predicted Price ($100k)')
    ax.legend(fontsize=9)

plt.suptitle('Actual vs Predicted House Prices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Residuals ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, name, color in zip(
    axes,
    [y_pred_lr, y_pred_gb],
    ['Linear Regression', 'Gradient Boosting'],
    [COLORS['lr'], COLORS['gb']]
):
    residuals = y_test - preds
    ax.scatter(preds, residuals, alpha=0.2, s=8, color=color, edgecolors='none')
    ax.axhline(0, color='black', linewidth=2, linestyle='--')
    ax.set_title(f'Residuals — {name}', fontweight='bold')
    ax.set_xlabel('Predicted Price ($100k)')
    ax.set_ylabel('Residual')

plt.suptitle('Residual Analysis (closer to 0 = better)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Importance (Gradient Boosting) ─────────────────────────────────────
importances = pd.Series(gb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.RdYlGn(importances.values / importances.max())
bars = ax.barh(importances.index, importances.values, color=colors, edgecolor='white')

for bar, val in zip(bars, importances.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Importance Score')
ax.set_title('Gradient Boosting — Feature Importance\n(House Price Prediction)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Metric Comparison Bar Chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
model_names = ['Lin. Reg.', 'Grad. Boost']
bar_colors  = [COLORS['lr'], COLORS['gb']]

metrics_data = {
    'MAE ($100k)' : [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_gb),
    ],
    'RMSE ($100k)': [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_gb)),
    ],
    'R²'          : [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_gb),
    ],
}

for ax, (metric, vals) in zip(axes, metrics_data.items()):
    bars = ax.bar(model_names, vals, color=bar_colors, edgecolor='white', width=0.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + max(vals)*0.01,
                f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)
    ax.set_title(metric, fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Results & Key Insights

In [ ]:
lr_mae  = mean_absolute_error(y_test, y_pred_lr)
gb_mae  = mean_absolute_error(y_test, y_pred_gb)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))
lr_r2   = r2_score(y_test, y_pred_lr)
gb_r2   = r2_score(y_test, y_pred_gb)
top_feat = importances.index[-1]

print(f"""
====================================================
RESULTS SUMMARY — House Price Prediction
====================================================

Linear Regression (Baseline):
  MAE    : ${lr_mae*100_000:,.0f}  (avg error per house)
  RMSE   : ${lr_rmse*100_000:,.0f}
  R²     : {lr_r2:.4f}

Gradient Boosting:
  MAE    : ${gb_mae*100_000:,.0f}
  RMSE   : ${gb_rmse*100_000:,.0f}
  R²     : {gb_r2:.4f}

🏆 Best model: Gradient Boosting
   MAE improvement over baseline: ${(lr_mae - gb_mae)*100_000:,.0f}
====================================================

KEY INSIGHTS:
1. '{top_feat}' is the #1 predictor of house price —
   wealthier neighbourhoods have significantly higher prices.

2. Location (Latitude/Longitude) ranks highly — coastal California
   areas command much higher prices than inland regions.

3. Gradient Boosting significantly outperforms Linear Regression
   because house prices have non-linear relationships with features.

4. The engineered features (RoomsPerPerson, BedroomRatio) add
   meaningful signal beyond raw room counts.

LIMITATIONS & NEXT STEPS:
  • Add neighbourhood/zip code features for finer location signal
  • Try XGBoost or LightGBM for further improvement
  • Use SHAP values for individual prediction explanations
  • Apply log transformation to target (price) to handle skewness
====================================================
""")